# 🚀 Cocopila Financial Data Agent Pipeline (Kaggle Bootstrap)

Notebook này chứa **toàn bộ mã nguồn Agent Pipeline** bao gồm:
1. **Môi trường & Phụ thuộc**: Khởi động Ollama Server & pull model `qwen2.5-coder:1.5b`.
2. **System Configuration, Provider & Utilities**: Cấu hình hệ thống, kết nối LLM, JSON repair utility.
3. **Prompts Mẫu (YAML Prompt Templates)**: Query Parser, Code Generator & Reflection Debugging.
4. **Agent State Definition**: Shared State Dictionary dùng trong LangGraph.
5. **Toàn bộ 5 Agent Pipeline Nodes**:
   - **Node 1: Query Parser** (Phân tích câu hỏi tài chính thành JSON cấu trúc)
   - **Node 2: Data Discovery** (Tìm kiếm bảng dữ liệu phù hợp với Search Engine & DataRegistry)
   - **Node 3: Schema Mapper** (Ánh xạ tiêu chí phụ sang tên cột thực tế trong CSV)
   - **Node 4: Code Generator & Reflection** (Sinh mã Python/Pandas trích xuất/tính toán/so sánh & tự động sửa lỗi)
   - **Node 5: AST Sandbox & Executor** (Thực thi mã Python an toàn trong Sandbox AST)
6. **Workflow StateGraph & Conditional Edge Routing**: Khởi tạo LangGraph app với vòng lặp Reflection Loop.
7. **Kiểm thử trực tiếp trên bộ câu hỏi ViFinQA**

In [ ]:
# 1. Cài đặt các gói phụ thuộc dự án (Không cần vLLM giúp tăng tốc độ cài đặt và tránh xung đột)
print("📥 Đang cài đặt dependencies...")
!pip install -q \
    langgraph>=0.2.0 \
    langchain-core>=0.3.0 \
    langchain-openai>=0.2.0 \
    pyyaml>=6.0 \
    json-repair>=0.30.0 \
    openpyxl>=3.1.0 \
    tabulate>=0.9.0 \
    thefuzz>=0.22.0


In [ ]:
# 2. Khởi động Ollama Server chạy nền & Tải model
import subprocess
import time
import requests
import os

print("🚀 Đang khởi động Ollama Server...")
ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

print("⏳ Chờ Ollama Server khởi động...")
for i in range(30):
    try:
        r = requests.get("http://localhost:11434/")
        if r.status_code == 200:
            print("✅ Ollama Server đã sẵn sàng tại port 11434!")
            break
    except Exception:
        time.sleep(1)
else:
    print("❌ Lỗi: Ollama Server không thể khởi động.")

MODEL_NAME = "qwen2.5-coder:1.5b"
print(f"📥 Đang tải mô hình {MODEL_NAME} từ Ollama registry...")
subprocess.run(["ollama", "pull", MODEL_NAME])
print(f"✅ Đã tải thành công mô hình {MODEL_NAME}!")

os.environ["MODEL_NAME"] = MODEL_NAME
os.environ["LLM_API_BASE"] = "http://localhost:11434/v1"
os.environ["LLM_API_KEY"] = "ollama"


## ⚙️ Section 1: System Configuration, LLM Provider & Utilities

In [ ]:
import os
import json
import logging
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Any, List, Optional, Literal, TypedDict
from json_repair import repair_json
from langchain_openai import ChatOpenAI
from langchain_core.language_models.chat_models import BaseChatModel

def is_kaggle_environment() -> bool:
    """Check if execution environment is Kaggle."""
    return os.path.exists("/kaggle/working")

@dataclass
class Config:
    """System configuration parameters."""
    MODEL_NAME: str = os.getenv("MODEL_NAME", "qwen2.5-coder:1.5b")
    LLM_API_BASE: str = os.getenv("LLM_API_BASE", "http://localhost:11434/v1")
    LLM_API_KEY: str = os.getenv("LLM_API_KEY", "ollama")
    TEMPERATURE: float = float(os.getenv("TEMPERATURE", "0.0"))
    MAX_TOKENS: int = int(os.getenv("MAX_TOKENS", "1024"))
    BASE_DIR: Path = Path("/kaggle/working/r2AI_2026/pipeline") if is_kaggle_environment() else Path.cwd()
    DATA_DIR: Path = Path(os.getenv("DATA_DIR", "/kaggle/working/r2AI_2026/pipeline/data"))
    MAX_RETRIES: int = int(os.getenv("MAX_RETRIES", "3"))

config = Config()

def get_llm(
    cfg: Optional[Config] = None,
    temperature: Optional[float] = None,
    max_tokens: Optional[int] = None,
) -> BaseChatModel:
    """Instantiate ChatOpenAI connected to local LLM endpoint."""
    cfg = cfg or config
    temp = temperature if temperature is not None else cfg.TEMPERATURE
    tokens = max_tokens if max_tokens is not None else cfg.MAX_TOKENS
    return ChatOpenAI(
        model=cfg.MODEL_NAME,
        openai_api_base=cfg.LLM_API_BASE,
        openai_api_key=cfg.LLM_API_KEY,
        temperature=temp,
        max_tokens=tokens,
        streaming=False,
    )

def safe_parse_json(content: str) -> Dict[str, Any]:
    """Parse JSON string with automatic repair fallback."""
    if not content or not content.strip():
        return {}
    cleaned = content.strip()
    if cleaned.startswith("```json"):
        cleaned = cleaned[7:]
    elif cleaned.startswith("```"):
        cleaned = cleaned[3:]
    if cleaned.endswith("```"):
        cleaned = cleaned[:-3]
    cleaned = cleaned.strip()

    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        pass

    try:
        repaired = repair_json(cleaned, return_objects=True)
        if isinstance(repaired, dict):
            return repaired
        elif isinstance(repaired, str):
            return json.loads(repaired)
        return {}
    except Exception:
        return {}


## 📝 Section 2: Prompts Mẫu (Prompt Templates)
Định nghĩa các Prompt mẫu cho Query Parser, Code Generator và Reflection Debugging Loop.

In [ ]:
PROMPT_QUERY_PARSER = {
    "system_prompt": """Bạn là chuyên gia phân tích câu hỏi tài chính Việt Nam.
Nhiệm vụ: Phân tích câu hỏi và trích xuất thành JSON cấu trúc bao gồm các trường:

1. "ten_cong_ty": Tên công ty hoặc mã chứng khoán được nhắc đến trong câu hỏi (nếu không có thì để "").
2. "so_nam": Danh sách các năm liên quan dưới dạng danh sách chuỗi (ví dụ: ["2023"], ["2021", "2022", "2023"]). Nếu câu hỏi đề cập khoảng năm (ví dụ: "từ năm 2021 đến năm 2023"), hãy liệt kê đầy đủ tất cả các năm trong khoảng đó.
3. "noi_dung": Nội dung hoặc chỉ tiêu tài chính cần tìm kiếm (ví dụ: "Doanh thu thuần", "Lợi nhuận sau thuế", "Tài sản ngắn hạn").
4. "thao_tac": Thao tác cần thực hiện, BẮT BUỘC chọn 1 trong 2 loại:
   - "trich_xuat": Trích xuất một hoặc nhiều giá trị / chỉ tiêu từ báo cáo tài chính
   - "so_sanh": So sánh cùng một số liệu giữa nhiều năm (theo từng năm) hoặc nhiều công ty
5. "tieu_chi_phu": Tiêu chí phụ bổ sung (ví dụ: mô tả tiêu chí theo từng năm thường chứa từ "đến" như "từ năm 2021 đến năm 2023", "31/12/2022", hoặc null nếu không có).

BẮT BUỘC CHỈ TRẢ VỀ JSON. KHÔNG GIẢI THÍCH.""",
    "json_schema": """{
  "ten_cong_ty": "tên công ty hoặc mã chứng khoán",
  "so_nam": ["2023"],
  "noi_dung": "nội dung cần tìm kiếm",
  "thao_tac": "trich_xuat | so_sanh",
  "tieu_chi_phu": "tiêu chí phụ hoặc null"
}""",
    "few_shot_examples": [
        {
            "user_query": "Doanh thu thuần năm 2023 của Vinamilk là bao nhiêu?",
            "parsed_output": '{\n  "ten_cong_ty": "Vinamilk",\n  "so_nam": ["2023"],\n  "noi_dung": "Doanh thu thuần",\n  "thao_tac": "trich_xuat",\n  "tieu_chi_phu": null\n}'
        },
        {
            "user_query": "So sánh lợi nhuận sau thuế của VCB từ năm 2021 đến năm 2023",
            "parsed_output": '{\n  "ten_cong_ty": "VCB",\n  "so_nam": ["2021", "2022", "2023"],\n  "noi_dung": "Lợi nhuận sau thuế",\n  "thao_tac": "so_sanh",\n  "tieu_chi_phu": "từ năm 2021 đến năm 2023"\n}'
        }
    ]
}

PROMPT_CODE_GENERATOR = {
    "system_prompt": """Bạn là một chuyên gia lập trình Python & Pandas Data Analysis.
Nhiệm vụ của bạn là sinh ra đoạn mã Python để truy vấn dữ liệu từ bảng báo cáo tài chính.

QUY TẮC BẮT BUỘC:
1. Dữ liệu nằm trong file CSV. Đọc file bằng pd.read_csv(file_path).
2. Cấu trúc bảng: Cột đầu tiên (label_column) chứa tên chỉ tiêu. Các cột sau chứa giá trị số.
3. Để tìm một chỉ tiêu, filter dòng bằng .str.contains trên cột label.
4. Giá trị số có thể ở dạng string với dấu chấm phân cách hàng nghìn (VD: '382.666') hoặc ngoặc cho số âm.
5. Luôn định nghĩa hàm clean_val(val) để parse giá trị số.
6. Kết quả cuối cùng PHẢI gán vào biến `result`.
7. CHỈ trả về code Python trong cặp ```python ... ```. KHÔNG giải thích.
""",
    "goal_descriptions": {
        "trich_xuat": "TRÍCH XUẤT giá trị cụ thể",
        "tinh_tong": "TÍNH TỔNG (tìm dòng Tổng/Cộng trước, nếu không có thì cộng các dòng con)",
        "so_sanh": "SO SÁNH giá trị giữa nhiều năm/công ty"
    },
    "goal_instructions": {
        "trich_xuat": """HƯỚNG DẪN CỤ THỂ (TRÍCH XUẤT):
1. Đọc file CSV bằng pd.read_csv(file_path).
2. Filter dòng chứa '{noi_dung}' ở cột '{label_col}'. Nếu rỗng, thử filter với từ khóa chính (VD: 'Tiền').
3. Lấy giá trị ở cột '{value_col}', clean bằng clean_val().
4. Gán vào biến result.""",
        "tinh_tong": """HƯỚNG DẪN CỤ THỂ (TÍNH TỔNG):
1. Đọc file CSV bằng pd.read_csv(file_path).
2. Tìm dòng có chứa 'Tổng' hoặc 'Cộng' hoặc tên chỉ tiêu '{noi_dung}' ở cột '{label_col}'.
3. Nếu tìm thấy → lấy giá trị ở cột '{value_col}', clean bằng clean_val(), gán vào result.
4. Nếu KHÔNG tìm thấy → tìm các dòng con riêng lẻ liên quan đến '{noi_dung}', cộng tổng giá trị và gán vào result.""",
        "so_sanh": """HƯỚNG DẪN CỤ THỂ (SO SÁNH NĂM / TÍNH TỐC ĐỘ TĂNG TRƯỞNG):
1. Đọc từng file CSV cho từng năm (ví dụ file_path_2019, file_path_2020, file_path_2021...).
2. Từ mỗi file, filter dòng chứa '{noi_dung}' (hoặc từ khóa chính như 'Tiền') ở cột '{label_col}'.
3. Lấy giá trị ở cột '{value_col}' của từng năm, clean bằng clean_val().
4. Tính tốc độ tăng trưởng phần trăm (%) giữa năm đầu và năm cuối: `growth_rate = ((val_last - val_first) / val_first) * 100`.
5. Gán kết quả vào `result` (ví dụ `result = growth_rate` hoặc dict chứa giá trị từng năm và tốc độ tăng trưởng)."""
    },
    "user_prompt_template": """Yêu cầu người dùng: {user_query}

MỤC TIÊU: {muc_tieu_desc}
NỘI DUNG cần tìm (ở cột label): '{noi_dung}'
Công ty: {ten_cong_ty}
Năm: {so_nam}
Tiêu chí phụ: {tieu_chi_phu}

DỮ LIỆU CÓ SẴN:
{files_context}

CỘT QUAN TRỌNG:
- Cột nhãn (chứa tên chỉ tiêu): '{label_col}'
- Cột giá trị: '{value_col}'

BIẾN ĐƯỜNG DẪN FILE:
{paths_str}

{goal_instruction}

🚨 BẮT BUỘC KHÔNG ĐƯỢC VI PHẠM:
1. Định nghĩa clean_val(val) để parse string số tài chính thành float.
2. Đọc file bằng pd.read_csv(file_path...).
3. Dùng `df['{label_col}'].astype(str).str.contains(..., case=False, na=False)` để lọc dòng. LUÔN DÙNG .astype(str) trước .str.
4. Kiểm tra `if not row.empty:` trước khi lấy `.values[0]`. Nếu rỗng, hãy lọc theo từ khóa ngắn hơn (ví dụ 'Tiền' thay vì cả câu dài) hoặc trả về 0.0.
5. KHÔNG ĐƯỢC filter theo `df['Ma_Doanh_Nghiep'] == ...` vì dữ liệu đã đúng công ty.
6. CHỈ ĐƯỢC SỬ DỤNG CÁC BIẾN ĐƯỜNG DẪN FILE ĐÃ ĐƯỢC ĐỊNH NGHĨA Ở TRÊN:
{paths_str}
7. Kết quả cuối cùng BẮT BUỘC lưu vào biến `result`.""",
    "few_shot_examples": [
        {
            "user_query": "Doanh thu thuần năm 2023 của FPT",
            "file_path": "data/FPT_2023.csv",
            "column_mapping": '{"label_column": "CHỈ TIÊU", "value_column": "Năm nay"}',
            "generated_code": """```python
import pandas as pd

def clean_val(val):
    if pd.isna(val) or str(val).strip() in ['-', '', 'nan', 'NaN']:
        return 0.0
    val_str = str(val).strip().replace('.', '')
    if '(' in val_str and ')' in val_str:
        val_str = '-' + val_str.replace('(', '').replace(')', '')
    return float(val_str)

df = pd.read_csv(file_path)
row = df[df['CHỈ TIÊU'].str.contains('Doanh thu thuần', case=False, na=False)]
raw_val = row['Năm nay'].values[0]
result = clean_val(raw_val)
```"""
        }
    ]
}

PROMPT_REFLECTION = {
    "system_prompt": """Đoạn mã Pandas trước đó đã gặp lỗi khi thực thi trong Sandbox môi trường.
Nhiệm vụ của bạn là phân tích lỗi Traceback, sửa lại đoạn mã Python/Pandas để đảm bảo không bị lỗi và giải quyết chính xác yêu cầu của người dùng.

QUY TẮC SỬA LỖI:
1. Phân tích nguyên nhân gây lỗi dựa vào error_traceback (ví dụ: KeyError tên cột, TypeError kiểu dữ liệu, v.v.).
2. Bọc xử lý ép kiểu hoặc làm sạch dữ liệu nếu cần (ví dụ: pd.to_numeric, pd.to_datetime, df[col].dropna()).
3. Kết quả cuối cùng BẮT BUỘC lưu vào biến `result`.
4. Trả về khối mã Python sửa đổi trong cặp markdown ```python ... ```.""",
    "user_prompt_template": """Yêu cầu người dùng: {user_query}

MỤC TIÊU: {muc_tieu}
NỘI DUNG: '{noi_dung}'
DỮ LIỆU:
{files_context}

CỘT: label='{label_col}', value='{value_col}'

BIẾN ĐƯỜNG DẪN:
{paths_str}

{sample_labels_str}

Mã Python bị lỗi trước đó:
```python
{previous_code}
```

Traceback Lỗi:
{error_traceback}

HƯỚNG DẪN SỬA LỖI:
1. Dùng `df['{label_col}'].astype(str).str.contains(..., case=False, na=False)`.
2. Kiểm tra `if not row.empty:` trước khi truy cập `.values[0]`. Nếu rỗng, thử dùng từ khóa ngắn gọn có trong danh sách chỉ tiêu thực tế ở trên.
3. Đảm bảo kết quả cuối cùng BẮT BUỘC lưu vào biến `result`."""
}


## 🧠 Section 3: Agent State Definition

In [ ]:
class AgentState(TypedDict, total=False):
    """Shared state dictionary passed across LangGraph nodes."""
    user_query: str
    parsed_query: Dict[str, Any]
    discovered_tables: List[Dict[str, Any]]
    column_mapping: Dict[str, str]
    generated_code: str
    execution_result: Any
    error_traceback: Optional[str]
    retry_count: int
    status: Literal["pending", "success", "error"]
    error_message: Optional[str]
    node_latencies: Dict[str, float]


## 🧩 Section 4: Agent Pipeline Nodes

### Node 1: Query Parser Node
Trích xuất `ten_cong_ty`, `so_nam`, `noi_dung`, `thao_tac`, `tieu_chi_phu` từ câu hỏi người dùng.

In [ ]:
import re
from langchain_core.messages import SystemMessage, HumanMessage

def _normalize_company_name(company_input: str, user_query: str) -> str:
    """Normalize company name or raw ticker input against rag_module/code_stock.csv map.
    
    Quy tắc:
    1. Load file rag_module/code_stock.csv.
    2. Nếu tên/mã đầu vào thuộc cột "Mã CK" -> giữ nguyên (không cần map).
    3. Nếu thuộc cột "Tên công ty" -> map sang "Mã CK" để thực hiện tra cứu.
    """
    company_input = company_input.strip() if company_input else ""
    user_query = user_query.strip() if user_query else ""

    try:
        from pathlib import Path
        import pandas as pd

        possible_paths = [
            Path("rag_module/code_stock.csv"),
            Path("rag_module/ViFinQA/code_stock.csv"),
            Path("/kaggle/working/r2AI_2026/rag_module/code_stock.csv"),
            Path("/kaggle/working/r2AI_2026/rag_module/ViFinQA/code_stock.csv"),
        ]

        csv_path = None
        for p in possible_paths:
            if p.exists():
                csv_path = p
                break

        name_to_code = []
        all_tickers = set()

        if csv_path:
            df = pd.read_csv(csv_path, encoding="utf-8-sig", dtype=str)
            ticker_col = next((c for c in df.columns if "CK" in c.upper()), df.columns[0])
            name_col = next((c for c in df.columns if "TÊN" in c.upper() or "TEN" in c.upper()), df.columns[1])

            for _, row in df.iterrows():
                code = str(row[ticker_col]).strip().upper() if pd.notna(row[ticker_col]) else ""
                name = str(row[name_col]).strip() if pd.notna(row[name_col]) else ""
                if code:
                    all_tickers.add(code)
                if code and name:
                    name_to_code.append((name, code))
        else:
            from rag_module.search_engine import _ensure_resources, _company_map
            _ensure_resources()
            if _company_map:
                name_to_code = list(_company_map)
                all_tickers = {code.upper() for _, code in _company_map}

        if company_input and company_input.upper() in all_tickers:
            return company_input.upper()

        name_to_code.sort(key=lambda x: len(x[0]), reverse=True)

        q_lower = user_query.lower()
        for name, code in name_to_code:
            if name.lower() in q_lower:
                return code

        if company_input:
            c_lower = company_input.lower()
            for name, code in name_to_code:
                if name.lower() in c_lower or c_lower in name.lower():
                    return code

        for w in re.findall(r"\b[A-Za-z]{3,5}\b", user_query):
            if w.upper() in all_tickers:
                return w.upper()
    except Exception as e:
        print(f"⚠️ [Query Parser] Stock code normalization error: {e}")
    return company_input

def _fallback_parse_query(user_query: str) -> Dict[str, Any]:
    q = user_query.strip()
    range_match = re.search(r"từ\s*(?:năm\s*)?(\d{4})\s*đến\s*(?:năm\s*)?(\d{4})", q, re.IGNORECASE)
    if range_match:
        y1, y2 = int(range_match.group(1)), int(range_match.group(2))
        start_y, end_y = min(y1, y2), max(y1, y2)
        years = [str(y) for y in range(start_y, end_y + 1)]
        tieu_chi_phu = range_match.group(0)
    else:
        years = re.findall(r"\b(20\d{2})\b", q)
        tieu_chi_phu = None

    q_lower = q.lower()
    if "so sánh" in q_lower or "thay đổi" in q_lower or "tăng trưởng" in q_lower or "từ năm" in q_lower or "đến năm" in q_lower:
        thao_tac = "so_sanh"
    else:
        thao_tac = "trich_xuat"

    m_ticker = re.search(r"\b([A-Z]{3,5})\b", q)
    company = m_ticker.group(1) if m_ticker else ""
    company = _normalize_company_name(company, user_query)

    clean_content = re.sub(r"\b(20\d{2})\b", "", q)
    if company:
        clean_content = re.sub(rf"\b{company}\b", "", clean_content, flags=re.IGNORECASE)
    
    stop_phrases = [
        "tốc độ tăng trưởng %", "tốc độ tăng trưởng", "tăng trưởng %", "tăng trưởng",
        "so sánh", "từ năm", "đến năm", "của", "năm", "báo cáo", "tài chính",
        "cho", "là bao nhiêu", "bao nhiêu"
    ]
    for word in stop_phrases:
        clean_content = re.sub(rf"\b{re.escape(word)}\b", "", clean_content, flags=re.IGNORECASE)
    
    clean_content = re.sub(r"\s+", " ", clean_content).strip()

    return {
        "ten_cong_ty": company,
        "so_nam": years,
        "noi_dung": clean_content or q,
        "thao_tac": thao_tac,
        "muc_tieu": thao_tac,
        "tieu_chi_phu": tieu_chi_phu,
    }

def parse_query_node(state: AgentState, cfg: Optional[Config] = None) -> AgentState:
    cfg = cfg or config
    start_time = time.time()
    user_query = state.get("user_query", "").strip()

    if not user_query:
        return {
            **state,
            "status": "error",
            "error_message": "User query is empty.",
            "parsed_query": {},
        }

    try:
        prompt_data = PROMPT_QUERY_PARSER
        system_prompt = prompt_data["system_prompt"]
        json_schema = prompt_data["json_schema"]
        few_shots = prompt_data.get("few_shot_examples", [])

        prompt_messages = [
            SystemMessage(content=f"{system_prompt}\n\nSchema Yêu cầu:\n{json_schema}")
        ]
        for example in few_shots:
            prompt_messages.append(HumanMessage(content=example["user_query"]))
            prompt_messages.append(SystemMessage(content=example["parsed_output"]))

        prompt_messages.append(HumanMessage(content=f"Câu hỏi: {user_query}"))

        llm = get_llm(cfg=cfg, temperature=0.0)
        response = llm.invoke(prompt_messages)
        raw_content = response.content if isinstance(response.content, str) else str(response.content)

        print(f"\n🔍 [Query Parser] Đang phân tích câu hỏi: '{user_query}'")
        think_match = re.search(r"<think>(.*?)</think>", raw_content, re.DOTALL)
        if think_match:
            print(f"💭 [Tư duy - Query Parser]:\n  {think_match.group(1).strip()}")

        parsed_json = safe_parse_json(raw_content)

        thao_tac = parsed_json.get("thao_tac") or parsed_json.get("muc_tieu") or "trich_xuat"
        if thao_tac not in ["trich_xuat", "so_sanh"]:
            thao_tac = "trich_xuat"
        
        parsed_json["thao_tac"] = thao_tac
        parsed_json["muc_tieu"] = thao_tac
        parsed_json["noi_dung"] = parsed_json.get("noi_dung", "")
        parsed_json["ten_cong_ty"] = _normalize_company_name(parsed_json.get("ten_cong_ty", ""), user_query)
        parsed_json["so_nam"] = parsed_json.get("so_nam", [])
        parsed_json["tieu_chi_phu"] = parsed_json.get("tieu_chi_phu")

        if isinstance(parsed_json["so_nam"], str):
            parsed_json["so_nam"] = [parsed_json["so_nam"]]

        print(
            f"📊 [Kết quả - Query Parser]:\n"
            f"   Công ty: {parsed_json.get('ten_cong_ty')}\n"
            f"   Năm: {parsed_json.get('so_nam')}\n"
            f"   Nội dung: {parsed_json.get('noi_dung')}\n"
            f"   Thao tác: {parsed_json.get('thao_tac')}\n"
            f"   Tiêu chí phụ: {parsed_json.get('tieu_chi_phu')}\n"
        )

        latency = time.time() - start_time
        node_latencies = state.get("node_latencies", {})
        node_latencies["query_parser"] = round(latency, 3)

        return {
            **state,
            "parsed_query": parsed_json,
            "status": "pending",
            "node_latencies": node_latencies,
        }

    except Exception as e:
        print(f"⚠️ [Query Parser] Lỗi LLM ({e}). Dùng Rule-based Fallback...")
        parsed_json = _fallback_parse_query(user_query)
        latency = time.time() - start_time
        node_latencies = state.get("node_latencies", {})
        node_latencies["query_parser"] = round(latency, 3)

        return {
            **state,
            "parsed_query": parsed_json,
            "status": "pending",
            "node_latencies": node_latencies,
        }


### Node 2: Data Discovery Node
Dùng `Search Engine` tra cứu bảng dữ liệu tương ứng theo công ty, năm, nội dung.

In [ ]:
def _resolve_csv_path(csv_path_str: str, cfg: Config) -> Optional[Path]:
    if not csv_path_str:
        return None
    p_str = csv_path_str.replace("\\", "/")
    direct = Path(p_str).resolve()
    if direct.exists():
        return direct
    idx_fin = p_str.find("ViFinQA")
    if idx_fin != -1:
        relative_part = p_str[idx_fin:]
        repo_root = Path("/kaggle/working/r2AI_2026").resolve()
        candidate1 = (repo_root / relative_part).resolve()
        if candidate1.exists():
            return candidate1
        candidate2 = (repo_root / "rag_module" / relative_part).resolve()
        if candidate2.exists():
            return candidate2
    return None

def _log_candidates(results: List[Dict[str, Any]], year_label: str = "") -> None:
    prefix = f" (Năm {year_label})" if year_label else ""
    print(f"   📋 Danh sách {len(results)} bảng ứng viên Top-K từ Search Engine{prefix}:")
    for idx, item in enumerate(results, 1):
        p_str = item.get("csv_path", "")
        file_name = Path(p_str).name if p_str else "N/A"
        ten_bang = item.get("Ten_Bang", "N/A")
        rrf = item.get("rrf_score", 0.0)
        print(f"      #{idx} RRF: {rrf:.6f} | File: {file_name} | Tên bảng: {ten_bang}")

def data_discovery_node(state: AgentState, cfg: Optional[Config] = None) -> AgentState:
    cfg = cfg or config
    start_time = time.time()

    user_query = state.get("user_query", "")
    parsed_query = state.get("parsed_query", {})

    ten_cong_ty = parsed_query.get("ten_cong_ty", "")
    so_nam = parsed_query.get("so_nam", [])
    noi_dung = parsed_query.get("noi_dung", "")
    thao_tac = parsed_query.get("thao_tac") or parsed_query.get("muc_tieu", "trich_xuat")

    if not so_nam and isinstance(user_query, str):
        so_nam = re.findall(r"\b(20\d{2})\b", user_query)
    if isinstance(user_query, str):
        try:
            from rag_module.search_engine import _resolve_ticker
            if ten_cong_ty:
                ten_cong_ty = _resolve_ticker(ten_cong_ty)
            else:
                m_ticker = re.search(r"\b([A-Za-z]{3,5})\b", user_query)
                if m_ticker:
                    ten_cong_ty = _resolve_ticker(m_ticker.group(1))
        except Exception:
            pass
    if not noi_dung:
        noi_dung = user_query if isinstance(user_query, str) else ""

    print(f"\n🔍 [Data Discovery] Bắt đầu tìm kiếm dữ liệu...")
    print(f"   - Công ty: '{ten_cong_ty}', Số năm: {so_nam}, Nội dung: '{noi_dung}'")

    report_type = "separate"
    if isinstance(user_query, str) and "hợp nhất" in user_query.lower():
        report_type = "consolidated"

    all_discovered_tables: List[Dict[str, Any]] = []

    try:
        from rag_module.search_engine import search_by_company_and_content
        import rag_module.search_engine as se
        se._ensure_resources()

        if not so_nam:
            results = search_by_company_and_content(
                company_name=ten_cong_ty, content=noi_dung, year=None, report_type=report_type, top_k=5
            )
            if results:
                _log_candidates(results)
                best_match = results[0]
                csv_path = _resolve_csv_path(best_match.get("csv_path", ""), cfg)
                if csv_path:
                    all_discovered_tables.append({
                        "csv_path": str(csv_path),
                        "Ten_Bang": best_match.get("Ten_Bang", ""),
                        "rrf_score": best_match.get("rrf_score", 0.0),
                        "Ma_Doanh_Nghiep": best_match.get("Ma_Doanh_Nghiep", ten_cong_ty),
                        "Nam_Tai_Chinh": best_match.get("Nam_Tai_Chinh", ""),
                        "Loai_Bao_Cao": best_match.get("Loai_Bao_Cao", report_type),
                    })
        else:
            for year in so_nam:
                results = search_by_company_and_content(
                    company_name=ten_cong_ty, content=noi_dung, year=str(year), report_type=report_type, top_k=5
                )
                if results:
                    _log_candidates(results, year_label=str(year))
                    best_match = results[0]
                    csv_path = _resolve_csv_path(best_match.get("csv_path", ""), cfg)
                    if csv_path:
                        all_discovered_tables.append({
                            "csv_path": str(csv_path),
                            "Ten_Bang": best_match.get("Ten_Bang", ""),
                            "rrf_score": best_match.get("rrf_score", 0.0),
                            "Ma_Doanh_Nghiep": best_match.get("Ma_Doanh_Nghiep", ten_cong_ty),
                            "Nam_Tai_Chinh": str(year),
                            "Loai_Bao_Cao": best_match.get("Loai_Bao_Cao", report_type),
                        })
    except Exception as e:
        print(f"⚠️ [Data Discovery] Lỗi Search Engine: {e}")

    latency = time.time() - start_time
    node_latencies = state.get("node_latencies", {})
    node_latencies["data_discovery"] = round(latency, 3)

    if not all_discovered_tables:
        return {
            **state,
            "status": "error",
            "error_message": "Không tìm thấy bảng dữ liệu phù hợp.",
            "discovered_tables": [],
            "node_latencies": node_latencies,
        }

    return {
        **state,
        "discovered_tables": all_discovered_tables,
        "matched_table_path": all_discovered_tables[0]["csv_path"],
        "status": "pending",
        "node_latencies": node_latencies,
    }


### Node 3: Schema Mapper Node
Ánh xạ `tieu_chi_phu` sang tên cột thực tế trong bảng CSV (rule-based fuzzy matching).

In [ ]:
import pandas as pd
from thefuzz import process, fuzz

DEFAULT_VALUE_COLUMNS = ["Năm nay", "Năm trước", "Số cuối năm", "Số đầu năm", "Số cuối kỳ", "Số đầu kỳ", "Kỳ này", "Kỳ trước"]
METADATA_HEADER_COLUMNS = ["Ma_Doanh_Nghiep", "Ten_Doanh_Nghiep", "Nam_Tai_Chinh", "Loai_Bao_Cao", "Ten_Bang", "Don_Vi_Tinh", "Tep_Nguon"]
KNOWN_LABEL_COLUMNS = ["CHÍ TIÊU", "CHỈ TIÊU", "TÀI SẢN", "NGUỒN VỐN", "Cột_0", "Chỉ tiêu", "Mã số", "STT"]

def _get_columns_from_table(table: Dict[str, Any]) -> List[str]:
    csv_path = table.get("csv_path", "")
    if not csv_path:
        return []
    try:
        df = pd.read_csv(csv_path, nrows=2)
        return list(df.columns)
    except Exception:
        return []

def _find_label_column(columns: List[str]) -> Optional[str]:
    for c in KNOWN_LABEL_COLUMNS:
        if c in columns:
            return c
    for c in columns:
        if c not in METADATA_HEADER_COLUMNS:
            return c
    return columns[0] if columns else None

def _find_value_column(columns: List[str], label_col: Optional[str] = None, tieu_chi_phu: Optional[str] = None) -> Optional[str]:
    label_idx = columns.index(label_col) if label_col and label_col in columns else -1
    value_candidate_cols = []
    for idx, c in enumerate(columns):
        if c in METADATA_HEADER_COLUMNS or c == label_col:
            continue
        if idx > label_idx or label_idx == -1:
            value_candidate_cols.append(c)

    if tieu_chi_phu and value_candidate_cols:
        clean_tcp = str(tieu_chi_phu).strip().lower()
        for col in value_candidate_cols:
            if clean_tcp in col.strip().lower():
                return col
        match, score = process.extractOne(tieu_chi_phu, value_candidate_cols, scorer=fuzz.token_set_ratio)
        if score >= 50:
            return match

    for c in DEFAULT_VALUE_COLUMNS:
        if c in value_candidate_cols:
            return c

    return value_candidate_cols[0] if value_candidate_cols else None

def schema_mapper_node(state: AgentState, cfg: Optional[Config] = None) -> AgentState:
    cfg = cfg or config
    start_time = time.time()
    parsed_query = state.get("parsed_query", {})
    discovered_tables = state.get("discovered_tables", [])
    tieu_chi_phu = parsed_query.get("tieu_chi_phu")

    print(f"\n🔍 [Schema Mapper] Đang ánh xạ tiêu chí → cột thực tế...")
    column_mapping: Dict[str, str] = {}

    if not discovered_tables:
        latency = time.time() - start_time
        node_latencies = state.get("node_latencies", {})
        node_latencies["schema_mapper"] = round(latency, 3)
        return {**state, "column_mapping": {}, "status": "pending", "node_latencies": node_latencies}

    first_table = discovered_tables[0]
    columns = _get_columns_from_table(first_table)
    label_col = _find_label_column(columns)
    if label_col:
        column_mapping["label_column"] = label_col
    value_col = _find_value_column(columns, label_col, tieu_chi_phu)
    if value_col:
        column_mapping["value_column"] = value_col
    column_mapping["all_columns"] = str(columns)

    print(f"📊 [Kết quả - Schema Mapper]: {column_mapping}\n")

    latency = time.time() - start_time
    node_latencies = state.get("node_latencies", {})
    node_latencies["schema_mapper"] = round(latency, 3)

    return {**state, "column_mapping": column_mapping, "status": "pending", "node_latencies": node_latencies}


### Node 4: Code Generator & Reflection Node
Sinh mã Pandas xử lý câu hỏi tài chính và hỗ trợ Reflection Debugging Loop khi xảy ra lỗi.

In [ ]:
def clean_python_code(raw_code: str) -> str:
    if not raw_code:
        return ""
    pattern = r"```(?:python)?\s*\n?(.*?)\n?```"
    matches = re.findall(pattern, raw_code, re.DOTALL)
    if matches:
        return matches[0].strip()
    cleaned = raw_code.strip()
    if cleaned.startswith("```") and cleaned.endswith("```"):
        cleaned = cleaned[3:-3].strip()
    return cleaned

def _build_files_context(discovered_tables: List[Dict[str, Any]], column_mapping: Dict[str, str]) -> str:
    if not discovered_tables:
        return "Không có bảng dữ liệu."
    lines = []
    for i, tbl in enumerate(discovered_tables):
        csv_path = tbl.get("csv_path", "")
        ten_bang = tbl.get("Ten_Bang", "N/A")
        nam = tbl.get("Nam_Tai_Chinh", "N/A")
        escaped_path = csv_path.replace('\\', '\\\\')
        lines.append(f"- File {i+1} (Năm {nam}):\n  Đường dẫn: '{escaped_path}'\n  Tên bảng: {ten_bang}\n")
    lines.append(f"\nColumn Mapping: {column_mapping}")
    return "\n".join(lines)

def code_generator_node(state: AgentState, cfg: Optional[Config] = None) -> AgentState:
    cfg = cfg or config
    start_time = time.time()

    user_query = state.get("user_query", "")
    parsed_query = state.get("parsed_query", {})
    discovered_tables = state.get("discovered_tables", [])
    column_mapping = state.get("column_mapping", {})
    error_traceback = state.get("error_traceback")
    retry_count = state.get("retry_count", 0)

    muc_tieu = parsed_query.get("muc_tieu", "trich_xuat")
    noi_dung = parsed_query.get("noi_dung", "")
    ten_cong_ty = parsed_query.get("ten_cong_ty", "")
    so_nam = parsed_query.get("so_nam", [])
    tieu_chi_phu = parsed_query.get("tieu_chi_phu")

    label_col = column_mapping.get("label_column", "CHỈ TIÊU")
    value_col = column_mapping.get("value_column", "Năm nay")

    files_context = _build_files_context(discovered_tables, column_mapping)

    paths_str = ""
    if discovered_tables:
        if len(discovered_tables) == 1:
            escaped = discovered_tables[0]["csv_path"].replace('\\', '\\\\')
            paths_str = f"file_path = '{escaped}'"
        else:
            for tbl in discovered_tables:
                nam = tbl.get("Nam_Tai_Chinh", "default")
                escaped = tbl["csv_path"].replace('\\', '\\\\')
                paths_str += f"file_path_{nam} = '{escaped}'\n"

    try:
        if not error_traceback or retry_count == 0:
            prompt_data = PROMPT_CODE_GENERATOR
            system_prompt = prompt_data["system_prompt"]
            few_shots = prompt_data.get("few_shot_examples", [])
            goal_descs = prompt_data.get("goal_descriptions", {})
            goal_instructions = prompt_data.get("goal_instructions", {})

            messages = [SystemMessage(content=system_prompt)]
            for ex in few_shots:
                messages.append(HumanMessage(content=f"Yêu cầu: {ex['user_query']}\nFile Path: {ex['file_path']}\nColumn Mapping: {ex['column_mapping']}"))
                messages.append(SystemMessage(content=ex["generated_code"]))

            goal_desc = goal_descs.get(muc_tieu, muc_tieu)
            goal_inst_template = goal_instructions.get(muc_tieu, "")
            goal_inst = goal_inst_template.format(
                noi_dung=noi_dung, label_col=label_col, value_col=value_col
            ) if goal_inst_template else ""

            user_template = prompt_data.get("user_prompt_template", "")
            human_content = user_template.format(
                user_query=user_query,
                muc_tieu_desc=goal_desc,
                noi_dung=noi_dung,
                ten_cong_ty=ten_cong_ty,
                so_nam=so_nam,
                tieu_chi_phu=tieu_chi_phu or "(không có)",
                files_context=files_context,
                label_col=label_col,
                value_col=value_col,
                paths_str=paths_str,
                goal_instruction=goal_inst,
            )

            messages.append(HumanMessage(content=human_content))
        else:
            prompt_data = PROMPT_REFLECTION
            system_prompt = prompt_data["system_prompt"]

            print(f"🔄 [Reflection Loop] Đang sửa lỗi mã nguồn (Lần {retry_count})...")
            sample_labels = []
            if discovered_tables:
                from pathlib import Path
                for tbl in discovered_tables:
                    c_path = tbl.get("csv_path")
                    if c_path and Path(c_path).exists():
                        try:
                            sub_df = pd.read_csv(c_path)
                            if label_col in sub_df.columns:
                                labels = sub_df[label_col].dropna().astype(str).head(20).tolist()
                                sample_labels.append(f"Mẫu chỉ tiêu thực tế trong file '{Path(c_path).name}':\n{labels}")
                        except Exception:
                            pass
            sample_labels_str = "\n\n".join(sample_labels) if sample_labels else ""

            user_template = prompt_data.get("user_prompt_template", "")
            human_content = user_template.format(
                user_query=user_query,
                muc_tieu=muc_tieu,
                noi_dung=noi_dung,
                files_context=files_context,
                label_col=label_col,
                value_col=value_col,
                paths_str=paths_str,
                sample_labels_str=sample_labels_str,
                previous_code=state.get('generated_code', ''),
                error_traceback=error_traceback,
            )
            messages = [SystemMessage(content=system_prompt), HumanMessage(content=human_content)]

        llm = get_llm(cfg=cfg, temperature=0.0)
        response = llm.invoke(messages)
        raw_text = response.content if isinstance(response.content, str) else str(response.content)

        think_match = re.search(r"<think>(.*?)</think>", raw_text, re.DOTALL)
        if think_match:
            print(f"💭 [Tư duy - Code Generator]:\n  {think_match.group(1).strip()}")

        code = clean_python_code(raw_text)
        print(f"📊 [Kết quả - Code Generator]:\n```python\n{code}\n```\n")

        latency = time.time() - start_time
        node_latencies = state.get("node_latencies", {})
        node_latencies["code_generator"] = round(latency, 3)

        return {**state, "generated_code": code, "status": "pending", "node_latencies": node_latencies}

    except Exception as e:
        latency = time.time() - start_time
        node_latencies = state.get("node_latencies", {})
        node_latencies["code_generator"] = round(latency, 3)
        return {**state, "status": "error", "error_message": f"Code generator error: {str(e)}", "node_latencies": node_latencies}


### Node 5: AST Sandbox & Executor Node
Thực thi mã Python trong môi trường Sandbox AST an toàn và thu thập kết quả `result`.

In [ ]:
import ast
import traceback
import numpy as np

class SecurityError(Exception):
    pass

FORBIDDEN_AST_NODES = (ast.Import, ast.ImportFrom)
FORBIDDEN_BUILTINS = {"eval", "exec", "__import__", "open", "compile", "globals", "locals", "input", "breakpoint"}
ALLOWED_MODULES = {"pandas", "pd", "numpy", "np", "datetime", "math", "re"}

def validate_ast(code_str: str) -> None:
    tree = ast.parse(code_str)
    for node in ast.walk(tree):
        if isinstance(node, ast.Import):
            for alias in node.names:
                if alias.name.split(".")[0] not in ALLOWED_MODULES:
                    raise SecurityError(f"Importing forbidden module: '{alias.name}'")
        elif isinstance(node, ast.ImportFrom):
            if node.module and node.module.split(".")[0] not in ALLOWED_MODULES:
                raise SecurityError(f"Importing from forbidden module: '{node.module}'")
        elif isinstance(node, ast.Call):
            if isinstance(node.func, ast.Name) and node.func.id in FORBIDDEN_BUILTINS:
                raise SecurityError(f"Call to forbidden function: '{node.func.id}'")

def format_result(result: Any) -> Any:
    if isinstance(result, pd.DataFrame):
        return {"type": "dataframe", "shape": list(result.shape), "columns": list(result.columns), "data": result.head(100).to_dict(orient="records")}
    elif isinstance(result, pd.Series):
        return {"type": "series", "name": str(result.name) if result.name else "result", "data": result.head(100).to_dict()}
    elif isinstance(result, (int, float, str, bool, list, dict)):
        return {"type": "scalar", "data": result}
    else:
        return {"type": "other", "data": str(result)}

def executor_node(state: AgentState, cfg: Optional[Config] = None) -> AgentState:
    cfg = cfg or config
    start_time = time.time()

    code_str = state.get("generated_code", "").strip()
    discovered_tables = state.get("discovered_tables", [])
    file_path = discovered_tables[0].get("csv_path", "") if discovered_tables else ""
    retry_count = state.get("retry_count", 0)

    if not code_str:
        return {**state, "status": "error", "error_traceback": "No code generated.", "retry_count": retry_count + 1}

    try:
        validate_ast(code_str)
        print(f"⚙️ [Executor] Đang thực thi mã Pandas...")

        df_loaded = None
        if file_path and os.path.exists(file_path):
            try:
                df_loaded = pd.read_csv(file_path)
            except Exception:
                pass

        exec_globals = {
            "pd": pd, "np": np, "pandas": pd, "numpy": np,
            "file_path": file_path, "df": df_loaded,
        }
        for tbl in discovered_tables:
            csv_p = tbl.get("csv_path", "")
            nam = tbl.get("Nam_Tai_Chinh", "")
            if csv_p and nam:
                exec_globals[f"file_path_{nam}"] = csv_p

        exec(code_str, exec_globals)
        result_val = exec_globals.get("result")
        if result_val is None:
            raise ValueError("Biến `result` không được tìm thấy sau khi thực thi mã.")

        formatted = format_result(result_val)
        print(f"✅ [Executor] Thực thi THÀNH CÔNG!")
        print(f"📊 [Kết quả - Executor]:\n{json.dumps(formatted, indent=4, ensure_ascii=False)}\n")

        latency = time.time() - start_time
        node_latencies = state.get("node_latencies", {})
        node_latencies["executor"] = round(latency, 3)

        return {**state, "execution_result": formatted, "error_traceback": None, "status": "success", "node_latencies": node_latencies}

    except Exception as e:
        latency = time.time() - start_time
        node_latencies = state.get("node_latencies", {})
        node_latencies["executor"] = round(latency, 3)
        tb_str = traceback.format_exc()
        return {**state, "status": "error", "error_traceback": tb_str, "retry_count": retry_count + 1, "node_latencies": node_latencies}


## 🌐 Section 5: LangGraph Workflow & Edge Routing
Xây dựng đồ thị trạng thái Agent StateGraph và thiết lập điều kiện Reflection Loop.

In [ ]:
from langgraph.graph import StateGraph, END

def route_after_discovery(state: AgentState) -> Literal["schema_mapper", "__end__"]:
    if state.get("status") == "error":
        return END
    return "schema_mapper"

def route_after_execution(state: AgentState, cfg: Optional[Config] = None) -> Literal["code_generator", "__end__"]:
    cfg = cfg or config
    status = state.get("status")
    retry_count = state.get("retry_count", 0)

    if status == "success":
        return END
    if status == "error" and retry_count < cfg.MAX_RETRIES:
        print(f"🔄 Reflection Loop Activated! Retrying code generation ({retry_count}/{cfg.MAX_RETRIES})...")
        return "code_generator"
    return END

def create_cocopila_graph(cfg: Optional[Config] = None):
    cfg = cfg or config
    workflow = StateGraph(AgentState)

    # Add Nodes
    workflow.add_node("query_parser", lambda s: parse_query_node(s, cfg))
    workflow.add_node("data_discovery", lambda s: data_discovery_node(s, cfg))
    workflow.add_node("schema_mapper", lambda s: schema_mapper_node(s, cfg))
    workflow.add_node("code_generator", lambda s: code_generator_node(s, cfg))
    workflow.add_node("executor", lambda s: executor_node(s, cfg))

    # Add Edges
    workflow.set_entry_point("query_parser")
    workflow.add_edge("query_parser", "data_discovery")
    workflow.add_conditional_edges("data_discovery", route_after_discovery, {"schema_mapper": "schema_mapper", END: END})
    workflow.add_edge("schema_mapper", "code_generator")
    workflow.add_edge("code_generator", "executor")
    workflow.add_conditional_edges("executor", lambda s: route_after_execution(s, cfg), {"code_generator": "code_generator", END: END})

    return workflow.compile()

print("✅ Đã khởi tạo thành công hàm create_cocopila_graph()!")


## 🧪 Section 6: Dataset Linking & Running Agent Test

In [ ]:
import shutil

# 1. Dò tìm và tạo symlink cho Kaggle Datasets
dataset_path = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'qdrant_local_db' in dirs:
        dataset_path = Path(root)
        break

if not dataset_path:
    fallback_path = Path('/kaggle/input/r2-ai-output')
    if fallback_path.exists():
        dataset_path = fallback_path

if dataset_path:
    print(f"✅ Đã tìm thấy dataset tại: {dataset_path}")
    repo_dir = Path("/kaggle/working/r2AI_2026")
    if repo_dir.exists():
        vifinqa_src = dataset_path / "ViFinQA"
        qdrant_src = dataset_path / "qdrant_local_db"
        bm25_src = dataset_path / "bm25_index.pkl"

        vifinqa_dst = repo_dir / "ViFinQA"
        qdrant_dst = repo_dir / "rag_module" / "qdrant_local_db"
        bm25_dst = repo_dir / "rag_module" / "bm25_index.pkl"

        for src, dst in [(vifinqa_src, vifinqa_dst), (qdrant_src, qdrant_dst), (bm25_src, bm25_dst)]:
            if dst.exists() or dst.is_symlink():
                if dst.is_symlink():
                    os.remove(dst)
                elif dst.is_dir():
                    shutil.rmtree(dst)
                else:
                    os.remove(dst)
            if src.exists():
                os.symlink(src, dst)
                print(f"✅ Đã tạo liên kết tượng trưng: {dst} -> {src}")


In [ ]:
# 2. Chuyển thư mục và đọc danh sách câu hỏi
import sys

repo_dir = Path("/kaggle/working/r2AI_2026")
if repo_dir.exists():
    os.chdir(str(repo_dir))
    if str(repo_dir) not in sys.path:
        sys.path.insert(0, str(repo_dir))

questions_file = repo_dir / "ViFinQA" / "questions" / "questions.jsonl"
questions = []
if questions_file.exists():
    with open(questions_file, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                questions.append(json.loads(line))
    print(f"✅ Đã tải {len(questions)} câu hỏi từ file {questions_file.name}")
else:
    print(f"❌ Không tìm thấy file câu hỏi tại {questions_file}")


In [ ]:
# 3. Khởi tạo Agent và chạy thử nghiệm ngẫu nhiên với 10 câu hỏi
import random

print("🤖 Đang khởi tạo Agent workflow...")
app = create_cocopila_graph()

random.seed(42)
test_questions = random.sample(questions, min(10, len(questions))) if questions else []

print(f"🚀 Bắt đầu kiểm thử với {len(test_questions)} câu hỏi ngẫu nhiên...\n" + "=" * 80)

results_summary = []

for idx, q_item in enumerate(test_questions, 1):
    q_id = q_item.get("id")
    q_text = q_item.get("question")
    print(f"\n[{idx}/{len(test_questions)}] ❓ Câu hỏi ID {q_id}: {q_text}")
    print("-" * 60)
    
    inputs = {
        "user_query": q_text,
        "retry_count": 0,
        "node_latencies": {},
        "status": "pending"
    }
    
    try:
        final_state = None
        for output in app.stream(inputs):
            for node_name, node_state in output.items():
                latency = node_state.get('node_latencies', {}).get(node_name, 'N/A')
                print(f"   📍 Node: [{node_name.upper()}] (Thời gian chạy: {latency}s)")
                if node_name == "executor":
                    final_state = node_state
        
        status = final_state.get("status") if final_state else "error"
        exec_res = final_state.get("execution_result") if final_state else None
        err_msg = final_state.get("error_message") if final_state else None
        err_tb = final_state.get("error_traceback") if final_state else None
        
        print(f"   🏁 Trạng thái kết thúc: {status.upper()}")
        if status == "success":
            data_preview = exec_res.get("data")
            print(f"   ✅ Thành công! Dữ liệu kết quả: {data_preview}")
            results_summary.append({"id": q_id, "question": q_text, "status": "success"})
        else: 
            print(f"   ❌ Thất bại! Chi tiết lỗi: {err_msg or err_tb or 'Lỗi không xác định'}")
            results_summary.append({"id": q_id, "question": q_text, "status": "failed", "error": err_msg or err_tb})
            
    except Exception as e:
        print(f"   💥 Lỗi ngoại lệ trong quá trình chạy: {e}")
        results_summary.append({"id": q_id, "question": q_text, "status": "error", "error": str(e)})

print("\n" + "=" * 80 + "\n📊 TỔNG HỢP KẾT QUẢ KIỂM THỬ:\n" + "=" * 80)
success_count = sum(1 for r in results_summary if r["status"] == "success")
print(f"Tổng số câu hỏi: {len(results_summary)} | Thành công: {success_count} | Thất bại: {len(results_summary) - success_count}")
for r in results_summary:
    status_emoji = "✅" if r["status"] == "success" else "❌"
    print(f"{status_emoji} ID {r['id']}: {r['status'].upper()}")
